# Premier League Elo Rating Model

Prospective Premier League Elo ratings with margin-of-victory adjustment, home-field advantage, offseason regression, and expanding-window parameter tuning.

Adapted from `NFL_elo_v3.ipynb`, preserving its ten sections, Elo update,
MOV adjustment, offseason regression, tuning grids, expanding validation,
margin mapping, rankings, and CSV exports.

**Football adaptations and inherited modeling choices**
- Seasons use the starting year: `2024` means 2024/25. Only Premier League matches are included.
- Full-time wins/draws/losses score 1/0.5/0; there is no NFL overtime weighting.
- For fidelity, `p_home` and `p_away` retain their original column names. They are
  complementary Elo expected scores (win + half a draw), **not** separate home/away
  win probabilities. This notebook does not fit a three-way draw model.
- The original tuning and accuracy/log-loss reports exclude draws, while draws
  still update ratings. These inherited binary diagnostics are not three-way
  probability validation. Grids are retained as starting choices, not established
  Premier League optima. Selecting grids after inspecting results can bias evaluation.
- The original entry rule is retained: unseen clubs start at 1000. Returning clubs
  retain ratings regressed during their absence; no Championship matches are modeled.
  Rankings show only clubs present in the loaded current-season data.
- Goal margins replace point margins. The fitted spread is a rounded model handicap,
  not bookmaker odds. Weekly projections use calendar weeks, not official matchweeks.
- Run cells in order. The repeated full grid searches can take substantial time.
  All data-loading code is contained in the setup cell. By default it reads
  `data/premier_league_matches.csv` from this project's working directory.
  Set `REFRESH_DATA = True` in that cell to download updates. Failed downloads
  fall back to the saved CSV and display a warning. The latest result date is
  printed so the data's age is visible. CSV outputs go to the working directory.


In [1]:
# Run once if dependencies are missing, then restart the kernel:
# %pip install numpy pandas scipy scikit-learn jupyter


## 1. Setup and Premier League Data


In [2]:
import io
import warnings
import time
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss

# Source and column definitions:
# https://www.football-data.co.uk/englandm.php
# https://www.football-data.co.uk/notes.txt
# All loading code is in this cell. False uses the existing saved CSV.
# Change to True to download newer results; a failed download uses saved data.
REFRESH_DATA = False
DATA_FILE = Path("data/premier_league_matches.csv")
FIRST_DATA_SEASON = 2003
today = pd.Timestamp.now(tz="Europe/London")
LAST_DATA_SEASON = today.year if today.month >= 7 else today.year - 1

def read_football_csv(url):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(3):
        try:
            with urlopen(request, timeout=30) as response:
                payload = response.read()
            break
        except (HTTPError, URLError, TimeoutError) as exc:
            if isinstance(exc, HTTPError) and exc.code not in {429, 500, 502, 503, 504}:
                raise
            if attempt == 2:
                raise
            print(f"Retrying {url}: {exc}", flush=True)
            time.sleep(2 ** attempt)
    try:
        text = payload.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = payload.decode("cp1252")  # Some historical files use Windows encoding.
    # Older season files contain surplus trailing commas. Selecting the
    # needed columns tolerates those empty fields without skipping matches.
    columns = {"Div", "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG"}
    return pd.read_csv(io.StringIO(text), usecols=lambda column: column in columns)

def normalize_matches(raw, season=None):
    raw = raw.loc[raw["Div"].eq("E0")].copy()
    raw = raw.dropna(subset=["Date", "HomeTeam", "AwayTeam"])
    out = pd.DataFrame(index=raw.index)
    out["gameday"] = pd.to_datetime(raw["Date"], dayfirst=True, format="mixed", errors="raise")
    out["gametime"] = raw["Time"].fillna("00:00") if "Time" in raw else "00:00"
    out["season"] = (season if season is not None else
                     out["gameday"].dt.year - (out["gameday"].dt.month < 7).astype(int))
    out["home_team"] = raw["HomeTeam"].str.strip()
    out["away_team"] = raw["AwayTeam"].str.strip()
    for target, source in [("home_score", "FTHG"), ("away_score", "FTAG")]:
        out[target] = pd.to_numeric(raw[source], errors="raise") if source in raw else np.nan
    out["location"] = "Home"
    out["game_type"] = "PL"
    out["week"] = out["gameday"].dt.strftime("%G-W%V")  # ISO calendar week
    out["game_id"] = (out["season"].astype(str) + "_" + out["home_team"] + "_" + out["away_team"])
    return out.reset_index(drop=True)

def download_games():
    season_frames = []
    for season in range(FIRST_DATA_SEASON, LAST_DATA_SEASON + 1):
        code = f"{season % 100:02d}{(season + 1) % 100:02d}"
        url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"
        try:
            frame = normalize_matches(read_football_csv(url), season)
        except HTTPError as exc:
            if season == LAST_DATA_SEASON and exc.code == 404:
                warnings.warn(f"Season {season}/{season + 1} is not published yet; using available seasons.")
                continue
            raise
        if frame.empty:
            raise ValueError(f"No Premier League rows in {url}")
        season_frames.append(frame)
        print(f"Loaded {season}/{season + 1}: {len(frame)} matches", flush=True)

    games = pd.concat(season_frames, ignore_index=True)
    # The free fixture feed covers published upcoming matches, not necessarily the full season.
    try:
        fixtures = normalize_matches(read_football_csv("https://www.football-data.co.uk/fixtures.csv"))
        fixtures = fixtures.loc[fixtures["season"].eq(LAST_DATA_SEASON)
                                & fixtures["gameday"].ge(today.tz_localize(None).normalize())]
        games = pd.concat([games, fixtures], ignore_index=True)
    except (HTTPError, URLError, TimeoutError, ValueError, KeyError) as exc:
        warnings.warn(f"Upcoming fixtures unavailable; historical ratings still work: {exc}")

    # Prefer completed results over any duplicate fixture entry.
    games["completed"] = games["home_score"].notna() & games["away_score"].notna()
    games = (games.sort_values("completed", ascending=False)
             .drop_duplicates("game_id", keep="first")
             .sort_values(["gameday", "gametime", "game_id"]).reset_index(drop=True))
    assert not games["game_id"].duplicated().any()
    return games


if REFRESH_DATA or not DATA_FILE.exists():
    try:
        games = download_games()
    except (OSError, URLError, ValueError, KeyError) as exc:
        if not DATA_FILE.exists():
            raise RuntimeError(
                "The data source could not be reached and no saved CSV is available. "
                "Run this cell from the project folder, which contains data/ "
                "and the previously downloaded matches."
            ) from exc
        warnings.warn(f"Download failed; using saved Premier League data: {exc}")
        games = pd.read_csv(DATA_FILE, parse_dates=["gameday"])
    else:
        DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
        games.to_csv(DATA_FILE, index=False)
        print(f"Downloaded and saved data to {DATA_FILE.resolve()}")
else:
    games = pd.read_csv(DATA_FILE, parse_dates=["gameday"])
    print(f"Loaded saved Premier League data from {DATA_FILE.resolve()}")

# Exclude expired unplayed fixtures from the saved snapshot.
games = games.loc[
    games["completed"] | games["gameday"].ge(today.tz_localize(None).normalize())
].reset_index(drop=True)
assert not games["game_id"].duplicated().any()
print("Latest completed result:", games.loc[games["completed"], "gameday"].max().date())
print(games.columns)
print(games.shape)
print(games.head())
print("Latest loaded season:", games["season"].max())
print("Unplayed fixtures available:", (~games["completed"]).sum())


Loaded saved Premier League data from C:\Users\schne\premiere-league-elo\data\premier_league_matches.csv
Latest completed result: 2026-10-12
Index(['gameday', 'gametime', 'season', 'home_team', 'away_team', 'home_score',
       'away_score', 'location', 'game_type', 'week', 'game_id', 'completed'],
      dtype='str')
(8801, 12)
     gameday gametime  season   home_team      away_team  home_score  \
0 2003-08-16    00:00    2003     Arsenal        Everton         2.0   
1 2003-08-16    00:00    2003  Birmingham      Tottenham         1.0   
2 2003-08-16    00:00    2003   Blackburn         Wolves         5.0   
3 2003-08-16    00:00    2003      Fulham  Middlesbrough         3.0   
4 2003-08-16    00:00    2003   Leicester    Southampton         2.0   

   away_score location game_type      week                     game_id  \
0         1.0     Home        PL  2003-W33        2003_Arsenal_Everton   
1         0.0     Home        PL  2003-W33   2003_Birmingham_Tottenham   
2         1.0  

In [3]:
completed = games.dropna(subset=["home_score", "away_score"])

counts = (
    completed.groupby("season")
    .size()
    .reindex(range(FIRST_DATA_SEASON, LAST_DATA_SEASON + 1), fill_value=0)
    .rename("completed_matches")
)

display(counts.to_frame())

assert counts.loc[FIRST_DATA_SEASON:LAST_DATA_SEASON - 1].eq(380).all(), (
    "At least one completed historical season is missing matches."
)

assert not completed.duplicated(
    ["season", "home_team", "away_team"]
).any(), "Duplicate match found."

print("Latest completed match:", completed["gameday"].max())

,completed_matches
season,
2003,380
2004,380
2005,380
2006,380
2007,380
2008,380
2009,380
2010,380
2011,380


Latest completed match: 2026-10-12 00:00:00


In [4]:
print(games['location'].unique())
print(games['season'].unique())

<ArrowStringArray>
['Home']
Length: 1, dtype: str
[2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016
 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026]


## 2. Core Elo Rating Functions


In [5]:
import numpy as np
import pandas as pd
from IPython.display import display


def null_coalesce(x, y):
    return y if x is None else x

def compress_exp_grid(min_val, max_val, n=10, curve=2):
    grid = np.unique(np.round(
        min_val + (max_val - min_val) *
        (np.exp(curve * np.linspace(0, 1, n)) - 1) / (np.exp(curve) - 1),2))
    return grid

def calc_mov_scale(games):
    mean_log_margin = np.log(
        (games["home_score"] - games["away_score"]).abs() + 1
    ).mean()
    return mean_log_margin

def pl_home_result(home_score, away_score):
    if pd.isna(home_score) or pd.isna(away_score):
        return np.nan
    return 1.0 if home_score > away_score else 0.0 if home_score < away_score else 0.5

def elo_season(games, K, ratings=None, scale=400, start_rating=1000,
               use_mov=True, mov_scale=1, use_home_adv=True, home_adv=0):

    ratings = {} if ratings is None else ratings.copy()
    n = len(games)

    if use_mov and (pd.isna(mov_scale) or mov_scale <= 0):
        mov_scale = 1
    if not use_home_adv:
        home_adv = 0

    elo_home_pre = np.zeros(n)
    elo_away_pre = np.zeros(n)
    elo_home_post = np.zeros(n)
    elo_away_post = np.zeros(n)
    p_home = np.zeros(n)

    for i, g in enumerate(games.to_dict("records")):

        hid = str(g["home_team"])
        aid = str(g["away_team"])

        Ra = ratings.get(hid, start_rating)
        Rb = ratings.get(aid, start_rating)

        elo_home_pre[i] = Ra
        elo_away_pre[i] = Rb

        game_home_adv = 0 if g["location"] == "Neutral" else home_adv

        p = 1 / (1 + 10 ** ((Rb - Ra - game_home_adv) / scale))
        p_home[i] = p

        y = pl_home_result(home_score=g["home_score"], away_score=g["away_score"])

        if pd.isna(y):
            raise ValueError(f"Invalid Premier League result coding for game_id: {g['game_id']}")

        goal_diff = abs(g["home_score"] - g["away_score"])
        if g["home_score"] > g["away_score"]:
            winner_elo_diff = (Ra + game_home_adv) - Rb
        elif g["home_score"] < g["away_score"]:
            winner_elo_diff = Rb - (Ra + game_home_adv)
        else:
            winner_elo_diff = 0

        if not use_mov:
          mov_mult = 1
        elif g["home_score"] == g["away_score"]:
          mov_mult = 1
        else:
          mov_mult = np.log(goal_diff + 1) * (1.74 / (0.001 * winner_elo_diff + 1.74)) / mov_scale

        delta = K * (y - p) * mov_mult

        ratings[hid] = Ra + delta
        ratings[aid] = Rb - delta

        elo_home_post[i] = ratings[hid]
        elo_away_post[i] = ratings[aid]

    games_out = games.copy()
    games_out["elo_home_pre"] = elo_home_pre
    games_out["elo_away_pre"] = elo_away_pre
    games_out["elo_home_post"] = elo_home_post
    games_out["elo_away_post"] = elo_away_post
    games_out["p_home"] = p_home
    games_out["p_away"] = 1 - p_home

    return {"games": games_out, "ratings": ratings}

### Single-Season Elo Sanity Check


In [6]:
# Sort it chronologically, calculate its MOV scale, and run
test_games = games[(games["season"] == 2024) & games["home_score"].notna() & games["away_score"].notna()].copy()

test_games = test_games.sort_values(["gameday", "game_id"]).reset_index(drop=True)
mov_scale = calc_mov_scale(test_games)
test_elo = elo_season(games=test_games, K=20, ratings={}, scale=400, start_rating=1000, use_mov=True, mov_scale=mov_scale, use_home_adv=True, home_adv=50)

test_elo["games"][["gameday", "home_team", "away_team", "elo_home_pre", "elo_away_pre", "p_home", "elo_home_post", "elo_away_post"]].head(20)
display(test_elo["ratings"])

#Sanity Check - average of elo values should still be around starting mean of 1000
print("Average ELO Rank: ",
      sum(test_elo["ratings"].values())
      / len(test_elo["ratings"]))

{'Man United': np.float64(943.1500249920762),
 'Fulham': np.float64(993.5904682120648),
 'Arsenal': np.float64(1118.9304401554816),
 'Wolves': np.float64(963.0282201633147),
 'Everton': np.float64(1018.1312261125747),
 'Brighton': np.float64(1047.8015897498879),
 'Ipswich': np.float64(828.3670093741906),
 'Liverpool': np.float64(1143.5231400698185),
 'Newcastle': np.float64(1061.4931592289806),
 'Southampton': np.float64(786.2275423492214),
 "Nott'm Forest": np.float64(1036.0023785951362),
 'Bournemouth': np.float64(1033.0218223237052),
 'West Ham': np.float64(951.7938289025835),
 'Aston Villa': np.float64(1056.9917204646215),
 'Brentford': np.float64(1032.2833583253355),
 'Crystal Palace': np.float64(1038.2165456522052),
 'Chelsea': np.float64(1071.1272388083462),
 'Man City': np.float64(1114.0335964522642),
 'Leicester': np.float64(832.9637103531071),
 'Tottenham': np.float64(929.3229797150854)}

Average ELO Rank:  1000.0


## 3. Offseason Regression and Evaluation


In [7]:
def carry_over(ratings, carry=0.6):
    if len(ratings) == 0:
        return ratings

    mu = np.mean(list(ratings.values()))
    return {team: carry * rating + (1 - carry) * mu
            for team, rating in ratings.items()}

def logloss(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log1p(-p))

### Carryover Sanity Check


In [8]:
#Sanity Checks
test_carry = carry_over(test_elo["ratings"], carry=0.75)

print(np.mean(list(test_carry.values())))
print(max(test_elo["ratings"].values()))
print(max(test_carry.values()))

1000.0
1143.5231400698185
1107.6423550523639


## 4. Elo Parameter Evaluation


In [9]:
def score_elo_params(all_games, seasons_tune, K, home_adv,
                     use_mov=True, mov_scale=1, use_home_adv=True,
                     scale=400, start_rating=1000, carry=0.6,
                     initial_ratings=None):

  ratings = {} if initial_ratings is None else initial_ratings.copy()
  ll_vec = []

  if not use_home_adv:
      home_adv = 0

  for s in seasons_tune:
      g_s = (all_games[all_games["season"] == s]
              .sort_values(["gameday", "game_id"])
              .reset_index(drop=True))

      ratings = carry_over(ratings, carry=carry)
      res = elo_season(games=g_s, ratings=ratings, K=K, scale=scale,
                       start_rating=start_rating, use_mov=use_mov,
                       mov_scale=mov_scale, use_home_adv=use_home_adv,
                       home_adv=home_adv)

      games_res = res["games"]
      non_ties = games_res["home_score"] != games_res["away_score"]

      p = games_res.loc[non_ties, "p_home"].to_numpy()
      y = (games_res.loc[non_ties, "home_score"] >
            games_res.loc[non_ties, "away_score"]).astype(float).to_numpy()

      ll_vec.extend(logloss(p, y))
      ratings = res["ratings"]

  return np.nanmean(ll_vec)

### Initial Parameter Test


In [10]:
test_score = score_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K=20,
    home_adv=50,
    use_mov=True,
    mov_scale=calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])]),
    use_home_adv=True,
    scale=400,
    start_rating=1000,
    carry=0.75
)

print(test_score)

0.5808563194663194


Compare the computed diagnostic above with the naive 50/50 baseline: $-\log(0.5)=0.6931$. No NFL performance result is carried over.


## 5. Hyperparameter Tuning


In [11]:
from itertools import product

def tune_elo_params(all_games, seasons_tune, K_grid, home_adv_grid,
                    use_mov=True, mov_scale=1, use_home_adv=True,
                    scale=400, start_rating=1000, carry=0.6,
                    use_carry_grid=True,
                    carryover_grid=(0.35, 0.50, 0.65, 0.80),
                    initial_ratings=None):

    if not use_home_adv:
        home_adv_grid = [0]

    carry_grid = carryover_grid if use_carry_grid else [carry]
    results = []

    for K, home_adv, carry_val in product(K_grid, home_adv_grid, carry_grid):
        ll = score_elo_params(
            all_games=all_games, seasons_tune=seasons_tune,
            K=K, home_adv=home_adv,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, scale=scale,
            start_rating=start_rating, carry=carry_val,
            initial_ratings=initial_ratings
        )

        results.append({"K": K, "home_adv": home_adv, "carry": carry_val,
                        "logloss": ll})

    results = pd.DataFrame(results).sort_values("logloss").reset_index(drop=True)

    best = results.iloc[0]

    return {"K_best": best["K"], "home_adv_best": best["home_adv"],
            "carry_best": best["carry"], "logloss": best["logloss"],
            "results": results}

### Tuning Sanity Check


In [12]:
K_grid = [10, 15, 20, 25]
home_adv_grid = [25, 40, 55, 70]

mov_scale = calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])])

test_tune = tune_elo_params(all_games=games, seasons_tune=[2021, 2022, 2023, 2024], K_grid=K_grid, 
                            home_adv_grid=home_adv_grid, mov_scale=mov_scale, carryover_grid=[0.50, 0.65, 0.75, 0.85])

print(test_tune["K_best"])
print(test_tune["home_adv_best"])
print(test_tune["carry_best"])
print(test_tune["logloss"])

test_tune["results"].head(10)

25.0
70.0
0.85
0.5732717904845079


,K,home_adv,carry,logloss
0,25,70,0.85,0.573272
1,25,55,0.85,0.573466
2,25,40,0.85,0.575300
3,25,70,0.75,0.575485
4,25,55,0.75,0.575633
5,25,40,0.75,0.577432
6,25,70,0.65,0.577882
7,20,70,0.85,0.577961
8,25,55,0.65,0.577982
9,20,55,0.85,0.578026


In [13]:
compress_exp_grid(1, 50, n=10, curve=-1)
compress_exp_grid(1, 60, n=10, curve=1)


array([ 1.  ,  5.04,  9.54, 14.58, 20.22, 26.51, 33.54, 41.4 , 50.18,
       60.  ])

## 6. Prospective Expanding-Window Elo


In [14]:
def run_expanding_k_elo(all_games, first_train_year=2003, first_eval_year=2010,
                        K_grid=None, home_adv_grid=None, scale=400,
                        start_rating=1000, carry=0.6, use_carry_grid=True,
                        carryover_grid=(0.35, 0.50, 0.65, 0.80),
                        param_tune_window=None, use_mov=True, use_home_adv=True,
                        projection_season=None):

    if K_grid is None:
        K_grid = compress_exp_grid(1, 60, n=10, curve=-1)
    if home_adv_grid is None:
        home_adv_grid = compress_exp_grid(1, 60, n=10, curve=1)
    if not use_home_adv:
        home_adv_grid = [0]

    K_grid = np.atleast_1d(K_grid)
    home_adv_grid = np.atleast_1d(home_adv_grid)
    fixed_param_mode = len(K_grid) == 1 and len(home_adv_grid) == 1 and not use_carry_grid

    seasons = sorted(all_games["season"].unique())
    eval_years = [s for s in seasons if s >= first_eval_year]

    # Include the current schedule season even if it has no completed games yet.
    if projection_season is not None and projection_season not in eval_years:
        eval_years = sorted(set(eval_years + [projection_season]))

    all_out = []
    param_out = []
    ratings = {}
    ratings_before_season = {}

    for yr in eval_years:
        train_years = [s for s in seasons if first_train_year <= s < yr]
        if len(train_years) == 0:
            raise ValueError(f"No training seasons available before eval year: {yr}")

        param_tune_years = (
            train_years if param_tune_window is None
            else train_years[-param_tune_window:]
        )

        train_games = all_games[all_games["season"].isin(train_years)]
        mov_scale = calc_mov_scale(train_games) if use_mov else np.nan

        if fixed_param_mode:
            K_best = K_grid[0]
            home_adv_best = home_adv_grid[0] if use_home_adv else 0
            carry_best = carry
            best_logloss = np.nan
        else:
            param_fit = tune_elo_params(
                all_games=all_games, seasons_tune=param_tune_years,
                K_grid=K_grid, home_adv_grid=home_adv_grid,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, scale=scale,
                start_rating=start_rating, carry=carry,
                use_carry_grid=use_carry_grid,
                carryover_grid=carryover_grid,
                initial_ratings=ratings_before_season.get(param_tune_years[0], {})
            )

            K_best = param_fit["K_best"]
            home_adv_best = param_fit["home_adv_best"] if use_home_adv else 0
            carry_best = param_fit["carry_best"] if use_carry_grid else carry
            best_logloss = param_fit["logloss"]

        # Bootstrap training history once; later seasons keep the prior final ratings.
        if yr == eval_years[0]:
            for s in train_years:
                season_games = (all_games[all_games["season"] == s]
                                .sort_values(["gameday", "gametime", "game_id"])
                                .reset_index(drop=True))

                ratings_before_season[s] = ratings.copy()
                ratings = carry_over(ratings, carry=carry_best)
                res = elo_season(
                    games=season_games, ratings=ratings, K=K_best,
                    scale=scale, start_rating=start_rating,
                    use_mov=use_mov, mov_scale=mov_scale,
                    use_home_adv=use_home_adv, home_adv=home_adv_best
                )

                ratings = res["ratings"]

        ratings_before_season[yr] = ratings.copy()
        ratings = carry_over(ratings, carry=carry_best)

        eval_games = (all_games[all_games["season"] == yr]
                      .sort_values(["gameday", "gametime", "game_id"])
                      .reset_index(drop=True))

        res_eval = elo_season(
            games=eval_games, ratings=ratings, K=K_best,
            scale=scale, start_rating=start_rating,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, home_adv=home_adv_best
        )

        ratings = res_eval["ratings"]
        games_out = res_eval["games"].copy()
        games_out["K_used"] = K_best
        games_out["home_adv_used"] = np.where(games_out["location"] == "Neutral", 0, home_adv_best)
        games_out["carry_used"] = carry_best
        games_out["mov_scale_used"] = mov_scale
        games_out["use_mov"] = use_mov
        games_out["use_home_adv"] = use_home_adv
        games_out["train_start"] = min(train_years)
        games_out["train_end"] = max(train_years)
        games_out["param_tune_start"] = min(param_tune_years)
        games_out["param_tune_end"] = max(param_tune_years)
        games_out["param_tune_n"] = len(param_tune_years)
        games_out["param_tune_window"] = param_tune_window

        all_out.append(games_out)

        param_out.append({
            "eval_year": yr,
            "train_start": min(train_years),
            "train_end": max(train_years),
            "param_tune_start": min(param_tune_years),
            "param_tune_end": max(param_tune_years),
            "param_tune_n": len(param_tune_years),
            "param_tune_window": param_tune_window,
            "K_best": K_best,
            "home_adv_best": home_adv_best,
            "carry_best": carry_best,
            "mov_scale": mov_scale,
            "use_mov": use_mov,
            "use_home_adv": use_home_adv,
            "use_carry_grid": use_carry_grid,
            "logloss": best_logloss
        })

    return {
        "games": pd.concat(all_out, ignore_index=True),
        "K_by_year": pd.DataFrame(param_out),
        "ratings": res_eval["ratings"],
        "current_season": eval_years[-1]
    }

### Prospective Validation Test


In [15]:
K_grid=[15, 20, 25, 30, 35, 40, 45, 50] 
[x - 5 for x in K_grid]

[10, 15, 20, 25, 30, 35, 40, 45]

In [16]:
# Keep completed Premier League matches
elo_df = games[games["game_type"].isin(["PL"]) &
               games["home_score"].notna() &
               games["away_score"].notna()].copy()

test_expanding = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2003,
    first_eval_year=2008,
    K_grid=[25, 35, 45, 55],
    home_adv_grid=[25, 35, 45, 55],
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.4, 0.5, 0.65, 0.75, 0.85],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True
)

#Inspect
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team",
                         "elo_home_pre", "elo_away_pre", "p_home", "K_used",
                         "location", "home_adv_used", "carry_used", "mov_scale_used"]])

#Sanity Check
test_expanding["K_by_year"][["eval_year", "train_start", "train_end", "param_tune_start", "param_tune_end"]]

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2008,2003,2007,2003,2007,5,5,25.0,55.0,0.85,0.713258,True,True,True,0.573159
1,2009,2003,2008,2004,2008,5,5,25.0,55.0,0.85,0.713323,True,True,True,0.555456
2,2010,2003,2009,2005,2009,5,5,25.0,55.0,0.85,0.719907,True,True,True,0.549275
3,2011,2003,2010,2006,2010,5,5,25.0,55.0,0.85,0.715855,True,True,True,0.556623
4,2012,2003,2011,2007,2011,5,5,25.0,55.0,0.85,0.720574,True,True,True,0.556934
5,2013,2003,2012,2008,2012,5,5,25.0,55.0,0.85,0.716412,True,True,True,0.565807
6,2014,2003,2013,2009,2013,5,5,25.0,55.0,0.85,0.724139,True,True,True,0.569851
7,2015,2003,2014,2010,2014,5,5,25.0,55.0,0.85,0.723250,True,True,True,0.579183
8,2016,2003,2015,2011,2015,5,5,25.0,55.0,0.85,0.721314,True,True,True,0.579146
9,2017,2003,2016,2012,2016,5,5,25.0,55.0,0.85,0.726444,True,True,True,0.571619


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,location,home_adv_used,carry_used,mov_scale_used
0,2008,2008-08-16,Arsenal,West Brom,1188.643076,932.673766,0.856937,25.0,Home,55.0,0.85,0.713258
1,2008,2008-08-16,Bolton,Stoke,970.237607,1000.000000,0.536256,25.0,Home,55.0,0.85,0.713258
2,2008,2008-08-16,Everton,Blackburn,1100.726877,1025.434141,0.679184,25.0,Home,55.0,0.85,0.713258
3,2008,2008-08-16,Hull,Fulham,1000.000000,958.515445,0.635390,25.0,Home,55.0,0.85,0.713258
4,2008,2008-08-16,Middlesbrough,Tottenham,1000.209680,1015.807610,0.556462,25.0,Home,55.0,0.85,0.713258
...,...,...,...,...,...,...,...,...,...,...,...,...
6895,2026,2026-10-10,Man United,Tottenham,1103.141023,930.275954,0.787797,35.0,Home,55.0,0.85,0.742128
6896,2026,2026-10-11,Crystal Palace,Nott'm Forest,974.384257,1034.052185,0.493283,35.0,Home,55.0,0.85,0.742128
6897,2026,2026-10-11,Hull,Everton,1022.801677,1054.059660,0.534114,35.0,Home,55.0,0.85,0.742128
6898,2026,2026-10-11,Liverpool,Man City,1117.590734,1250.210011,0.390119,35.0,Home,55.0,0.85,0.742128


,eval_year,train_start,train_end,param_tune_start,param_tune_end
0,2008,2003,2007,2003,2007
1,2009,2003,2008,2004,2008
2,2010,2003,2009,2005,2009
3,2011,2003,2010,2006,2010
4,2012,2003,2011,2007,2011
5,2013,2003,2012,2008,2012
6,2014,2003,2013,2009,2013
7,2015,2003,2014,2010,2014
8,2016,2003,2015,2011,2015
9,2017,2003,2016,2012,2016


In [17]:
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team", "elo_home_pre", "elo_away_pre", "p_home", "K_used", "location", "home_adv_used", "carry_used", "mov_scale_used"]])

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2008,2003,2007,2003,2007,5,5,25.0,55.0,0.85,0.713258,True,True,True,0.573159
1,2009,2003,2008,2004,2008,5,5,25.0,55.0,0.85,0.713323,True,True,True,0.555456
2,2010,2003,2009,2005,2009,5,5,25.0,55.0,0.85,0.719907,True,True,True,0.549275
3,2011,2003,2010,2006,2010,5,5,25.0,55.0,0.85,0.715855,True,True,True,0.556623
4,2012,2003,2011,2007,2011,5,5,25.0,55.0,0.85,0.720574,True,True,True,0.556934
5,2013,2003,2012,2008,2012,5,5,25.0,55.0,0.85,0.716412,True,True,True,0.565807
6,2014,2003,2013,2009,2013,5,5,25.0,55.0,0.85,0.724139,True,True,True,0.569851
7,2015,2003,2014,2010,2014,5,5,25.0,55.0,0.85,0.723250,True,True,True,0.579183
8,2016,2003,2015,2011,2015,5,5,25.0,55.0,0.85,0.721314,True,True,True,0.579146
9,2017,2003,2016,2012,2016,5,5,25.0,55.0,0.85,0.726444,True,True,True,0.571619


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,location,home_adv_used,carry_used,mov_scale_used
0,2008,2008-08-16,Arsenal,West Brom,1188.643076,932.673766,0.856937,25.0,Home,55.0,0.85,0.713258
1,2008,2008-08-16,Bolton,Stoke,970.237607,1000.000000,0.536256,25.0,Home,55.0,0.85,0.713258
2,2008,2008-08-16,Everton,Blackburn,1100.726877,1025.434141,0.679184,25.0,Home,55.0,0.85,0.713258
3,2008,2008-08-16,Hull,Fulham,1000.000000,958.515445,0.635390,25.0,Home,55.0,0.85,0.713258
4,2008,2008-08-16,Middlesbrough,Tottenham,1000.209680,1015.807610,0.556462,25.0,Home,55.0,0.85,0.713258
...,...,...,...,...,...,...,...,...,...,...,...,...
6895,2026,2026-10-10,Man United,Tottenham,1103.141023,930.275954,0.787797,35.0,Home,55.0,0.85,0.742128
6896,2026,2026-10-11,Crystal Palace,Nott'm Forest,974.384257,1034.052185,0.493283,35.0,Home,55.0,0.85,0.742128
6897,2026,2026-10-11,Hull,Everton,1022.801677,1054.059660,0.534114,35.0,Home,55.0,0.85,0.742128
6898,2026,2026-10-11,Liverpool,Man City,1117.590734,1250.210011,0.390119,35.0,Home,55.0,0.85,0.742128


### Out-of-Sample Performance


In [18]:
#Calculate out-of-sample performance for all evaluation seasons (draws excluded)
eval_games = test_expanding["games"].copy()
non_ties = eval_games["home_score"] != eval_games["away_score"]

eval_games["actual_home_win"] = (
    eval_games["home_score"] > eval_games["away_score"]).astype(int)

eval_games["pred_home_win"] = (eval_games["p_home"] >= 0.5).astype(int)

eval_games["logloss"] = np.nan
eval_games.loc[non_ties, "logloss"] = logloss(
    eval_games.loc[non_ties, "p_home"],
    eval_games.loc[non_ties, "actual_home_win"]
)

performance_by_year = (eval_games.loc[non_ties].groupby("season").agg(
    games=("game_id", "size"),
    accuracy=("pred_home_win",
              lambda x: (x == eval_games.loc[x.index, "actual_home_win"]).mean()),
    logloss=("logloss", "mean")).reset_index())

print(performance_by_year)

    season  games  accuracy   logloss
0     2008    283  0.734982  0.536578
1     2009    284  0.714789  0.541306
2     2010    269  0.657993  0.607936
3     2011    287  0.682927  0.584243
4     2012    272  0.716912  0.560569
5     2013    302  0.754967  0.557589
6     2014    287  0.672474  0.587615
7     2015    273  0.655678  0.607219
8     2016    296  0.746622  0.547660
9     2017    281  0.736655  0.541983
10    2018    309  0.718447  0.535934
11    2019    288  0.694444  0.574733
12    2020    297  0.639731  0.635172
13    2021    292  0.726027  0.558912
14    2022    293  0.699659  0.568630
15    2023    298  0.738255  0.542673
16    2024    287  0.696864  0.574694
17    2025    276  0.670290  0.604697
18    2026     41  0.658537  0.605898


In [19]:
overall_accuracy = (eval_games.loc[non_ties, "pred_home_win"] ==
                    eval_games.loc[non_ties, "actual_home_win"]).mean()

overall_logloss = eval_games.loc[non_ties, "logloss"].mean()

print("Accuracy:", overall_accuracy)
print("Log Loss:", overall_logloss)

Accuracy: 0.7033557046979866
Log Loss: 0.5703658514829357


In [20]:
accuracy_by_year = (
    eval_games.loc[non_ties]
    .groupby("season")
    .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())
    .reset_index(name="accuracy")
)

accuracy_by_year

,season,accuracy
0,2008,0.734982
1,2009,0.714789
2,2010,0.657993
3,2011,0.682927
4,2012,0.716912
5,2013,0.754967
6,2014,0.672474
7,2015,0.655678
8,2016,0.746622
9,2017,0.736655


## 7. Final Prospective Elo Model


In [ ]:
display(np.round(compress_exp_grid(15, 60, n=15, curve=1)))
display(np.round(compress_exp_grid(15, 80, n=15, curve=1)))
display(np.round(compress_exp_grid(40, 80, n=15, curve=1)*.01, 3))

array([45., 49., 52., 54., 57., 58., 60.])

array([25., 32., 39., 44., 49., 53., 57., 60.])

array([30., 36., 42., 47., 51., 54., 57., 60.])

array([0.4  , 0.427, 0.458, 0.492, 0.53 , 0.573, 0.621, 0.674, 0.733,
       0.8  ])

In [22]:
pl_elo = run_expanding_k_elo(
    all_games=elo_df, first_train_year=2003, first_eval_year=2008,
    K_grid=np.round(compress_exp_grid(15, 60, n=15, curve=1)),
    home_adv_grid=np.round(compress_exp_grid(15, 80, n=15, curve=1)),
    scale=400, start_rating=1000, use_carry_grid=True,
    carryover_grid=np.round(compress_exp_grid(40, 80, n=15, curve=1)*.01, 3),
    param_tune_window=5, use_mov=True, use_home_adv=True
)


### Performance Check


In [23]:
eval_games = pl_elo["games"].query("home_score != away_score").copy()

eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

Accuracy: 0.7069990412272291
Log Loss: 0.5702929003850818


In [24]:
performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

,season,games,accuracy,logloss
0,2008,283,0.734982,0.532695
1,2009,284,0.728873,0.529150
2,2010,269,0.687732,0.601030
3,2011,287,0.689895,0.583695
4,2012,272,0.713235,0.559760
5,2013,302,0.745033,0.558746
6,2014,287,0.710801,0.584489
7,2015,273,0.652015,0.613654
8,2016,296,0.750000,0.540331
9,2017,281,0.750890,0.537675


### Tuned Parameter Selection


In [25]:
params = pl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2008,32.0,80.0,0.8,0.713258,0.565650
1,2009,29.0,80.0,0.8,0.713323,0.547849
2,2010,29.0,80.0,0.8,0.719907,0.541263
3,2011,21.0,80.0,0.8,0.715855,0.548109
4,2012,24.0,80.0,0.8,0.720574,0.550168
5,2013,19.0,80.0,0.8,0.716412,0.559040
6,2014,19.0,80.0,0.8,0.724139,0.564734
7,2015,17.0,80.0,0.8,0.723250,0.575928
8,2016,24.0,73.0,0.8,0.721314,0.579285
9,2017,24.0,80.0,0.8,0.726444,0.570790


In [26]:
print("K values:")
print(params["K_best"].value_counts().sort_index())

print("\nHome advantage values:")
print(params["home_adv_best"].value_counts().sort_index())

print("\nCarry values:")
print(params["carry_best"].value_counts().sort_index())
params[["K_best", "home_adv_best", "carry_best"]].describe()

display(np.median(params["K_best"]))

K values:
K_best
17.0    1
19.0    2
21.0    1
24.0    4
26.0    5
29.0    4
32.0    2
Name: count, dtype: int64

Home advantage values:
home_adv_best
49.0     1
54.0     3
66.0     2
73.0     1
80.0    12
Name: count, dtype: int64

Carry values:
carry_best
0.8    19
Name: count, dtype: int64


np.float64(26.0)

### Rerun Elo Tuning (Tightened Grids)


In [30]:
display(np.round(compress_exp_grid(15, 60, n=15, curve=1)))
display(np.round(compress_exp_grid(15, 80, n=15, curve=1)))
display(np.round(compress_exp_grid(70, 95, n=10, curve=1)*.01, 3))

array([15., 17., 19., 21., 24., 26., 29., 32., 35., 39., 42., 46., 51.,
       55., 60.])

array([15., 18., 21., 24., 28., 31., 35., 40., 44., 49., 54., 60., 66.,
       73., 80.])

array([0.7  , 0.717, 0.736, 0.758, 0.781, 0.808, 0.838, 0.871, 0.908,
       0.95 ])

In [31]:
projection_season = int(games.loc[games["game_type"].isin(["PL"]),"season"].max())

pl_elo = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2003,
    first_eval_year=2008,
    K_grid=[15, 18, 21, 24, 27, 30, 33, 36],
    home_adv_grid=[45, 50, 55, 65, 70, 75, 80, 85, 95, 100],
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True,
    projection_season=projection_season
)


In [32]:
eval_games = pl_elo["games"].query("home_score != away_score").copy()
eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

Accuracy: 0.7079578139980824
Log Loss: 0.565636565112149


,season,games,accuracy,logloss
0,2008,283,0.749117,0.520856
1,2009,284,0.728873,0.519867
2,2010,269,0.687732,0.592813
3,2011,287,0.693380,0.581106
4,2012,272,0.724265,0.549223
5,2013,302,0.745033,0.557552
6,2014,287,0.703833,0.577993
7,2015,273,0.644689,0.624806
8,2016,296,0.760135,0.525996
9,2017,281,0.740214,0.534022


In [33]:
params = pl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]].rename(columns = {"logloss":"tuning_logloss"})

,eval_year,K_best,home_adv_best,carry_best,mov_scale,tuning_logloss
0,2008,27.0,100.0,1.0,0.713258,0.559084
1,2009,24.0,100.0,1.0,0.713323,0.538793
2,2010,24.0,100.0,1.0,0.719907,0.531665
3,2011,18.0,100.0,1.0,0.715855,0.537950
4,2012,18.0,100.0,1.0,0.720574,0.541832
5,2013,15.0,100.0,1.0,0.716412,0.550232
6,2014,15.0,100.0,1.0,0.724139,0.558657
7,2015,15.0,85.0,1.0,0.723250,0.569618
8,2016,18.0,70.0,1.0,0.721314,0.575322
9,2017,15.0,80.0,1.0,0.726444,0.564918


## 8. Elo Expected Score to Spread Map

Validate the Elo-to-spread mapping prospectively with an expanding window.

For each evaluation season, the spread coefficient is fit only on completed games from earlier seasons, then applied to that season's already-prospective Elo expected scores. This produces a leakage-safe historical goal-margin validation.

After validation, a final production coefficient is fit on all completed historical games and used only for future projections.


In [34]:
from scipy.optimize import minimize_scalar

spread_games = pl_elo["games"].dropna(subset=["home_score", "away_score", "p_home"]).copy()

spread_games = spread_games.sort_values(["season", "gameday", "gametime", "game_id"]).reset_index(drop=True)

spread_games["home_margin"] = (spread_games["home_score"] - spread_games["away_score"])

p = spread_games["p_home"].clip(1e-6, 1 - 1e-6)
spread_games["logit_p_home"] = np.log(p / (1 - p))

spread_seasons = sorted(spread_games["season"].unique())
spread_validation = []

for eval_season in spread_seasons[1:]:
    train = spread_games[spread_games["season"] < eval_season].copy()
    test = spread_games[spread_games["season"] == eval_season].copy()

    spread_opt = minimize_scalar(
        lambda slope: np.mean(
            np.abs(
                train["home_margin"]
                - slope * train["logit_p_home"]
            )
        ),
        bounds=(0, 20),
        method="bounded"
    )

    slope = spread_opt.x

    test["spread_slope_used"] = slope
    test["elo_home_margin"] = slope * test["logit_p_home"]
    test["model_home_spread"] = -(np.round(test["elo_home_margin"] * 2) / 2)
    test["spread_abs_error"] = (test["home_margin"] - test["elo_home_margin"]).abs()

    spread_validation.append(test)

spread_validation = pd.concat(spread_validation, ignore_index=True)

print("Prospective spread MAE:", spread_validation["spread_abs_error"].mean())
print("Fraction actual margin above predicted margin:", (spread_validation["home_margin"] > spread_validation["elo_home_margin"]).mean())


Prospective spread MAE: 1.2920149368591207
Fraction actual margin above predicted margin: 0.48726993865030677


In [35]:
spread_performance_by_year = (spread_validation.groupby("season").agg(games=("game_id", "size"), spread_slope_used=("spread_slope_used", "first"),
                                                                      spread_mae=("spread_abs_error", "mean")).reset_index())

display(spread_performance_by_year)

,season,games,spread_slope_used,spread_mae
0,2009,380,0.653522,1.277203
1,2010,380,0.784870,1.239994
2,2011,380,0.702600,1.324120
3,2012,380,0.714493,1.150546
4,2013,380,0.690902,1.330093
5,2014,380,0.714492,1.171662
6,2015,380,0.706396,1.254636
7,2016,380,0.696595,1.321671
8,2017,380,0.723853,1.274107
9,2018,380,0.732117,1.291923


### Production Spread Coefficient

Fit one final coefficient on all completed historical games. This coefficient is not used to score the historical validation above; it is used only for future projections.


In [36]:
production_spread_opt = minimize_scalar(
    lambda slope: np.mean(
        np.abs(
            spread_games["home_margin"]
            - slope * spread_games["logit_p_home"]
        )
    ),
    bounds=(0, 20),
    method="bounded"
)

spread_slope = production_spread_opt.x

print(f"Production spread slope: {spread_slope:.4f}")


Production spread slope: 0.7281


## 9. Current Elo Ratings

Use the final expanding fit's ratings directly, restricted to clubs in the latest
loaded season. The training history starts in 2003/04 and evaluation in 2008/09.


In [37]:
current_season = pl_elo["current_season"]
params = pl_elo["K_by_year"].set_index("eval_year").loc[current_season]

# Use the final expanding fit's current Elo state directly.
ratings = pl_elo["ratings"].copy()

current_teams = pd.unique(
    games.loc[
        games["season"] == current_season,
        ["home_team", "away_team"]
    ].values.ravel()
)

# Preserve the original notebook entry rule: a newly appearing team starts at 1000.
ratings = {team: ratings.get(team, 1000) for team in current_teams}

current_ratings = (
    pd.Series(ratings, name="elo")
    .rename_axis("team")
    .reset_index()
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)
current_ratings.insert(0, "rank", range(1, len(current_ratings) + 1))

current_ratings


,rank,team,elo
0,1,Man City,1352.453045
1,2,Arsenal,1318.928309
2,3,Brighton,1208.499764
3,4,Man United,1177.508628
4,5,Liverpool,1167.033587
5,6,Brentford,1154.047019
6,7,Bournemouth,1124.181524
7,8,Chelsea,1119.950603
8,9,Leeds,1114.348558
9,10,Aston Villa,1113.890025


## 10. Daily Rankings and Projections

Use current Elo ratings to identify games scheduled for a selected date and calculate each team's pregame Elo and expected score.

Projections cover only fixtures supplied by the feed. Times and match dates follow the source; daily selection uses Europe/London. `p_home` is an expected score, not a three-way home-win probability.


In [38]:
as_of_date = pd.Timestamp.now(tz="Europe/London").date()
rank_map = current_ratings.set_index("team")["rank"]

future_projections = games[
    (games["season"] == current_season) &
    games["game_type"].isin(["PL"]) &
    games["home_score"].isna() &
    games["away_score"].isna()
].copy()

future_projections["home_elo"] = future_projections["home_team"].map(ratings)
future_projections["away_elo"] = future_projections["away_team"].map(ratings)
future_projections["home_rank"] = future_projections["home_team"].map(rank_map)
future_projections["away_rank"] = future_projections["away_team"].map(rank_map)

future_projections["game_home_adv"] = np.where(future_projections["location"] == "Neutral", 0, params["home_adv_best"])
future_projections["p_home"] = 1 / (1 + 10 ** ((future_projections["away_elo"] - future_projections["home_elo"] - future_projections["game_home_adv"]) / 400))
future_projections["p_away"] = 1 - future_projections["p_home"]

projection_p = future_projections["p_home"].clip(1e-6, 1 - 1e-6)
future_projections["elo_home_margin"] = spread_slope * np.log(projection_p / (1 - projection_p))
future_projections["model_home_spread"] = -(np.round(future_projections["elo_home_margin"] * 2) / 2)
future_projections["as_of_date"] = as_of_date

future_projections = future_projections[["as_of_date", "week", "gameday", "gametime", "away_team", "home_team", "away_rank", "home_rank", "away_elo", "home_elo", "p_away", "p_home", "elo_home_margin", "model_home_spread"]
                                        ].sort_values(["gameday", "gametime"]).reset_index(drop=True)

daily_projections = future_projections[pd.to_datetime(future_projections["gameday"]).dt.date == as_of_date].reset_index(drop=True)

current_week = future_projections.iloc[0]["week"] if not future_projections.empty else None
weekly_projections = future_projections[future_projections["week"] == current_week].reset_index(drop=True)

daily_rankings = current_ratings.assign(as_of_date=as_of_date)

display(future_projections)    # every available upcoming fixture using current Elo
display(daily_projections)     # today's games
display(weekly_projections)    # current/upcoming calendar week
display(daily_rankings)        # current league-wide rankings


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-10-19,2026-W43,2026-10-19,20:00,Coventry,Tottenham,17,18,1008.600348,999.300689,0.420521,0.579479,0.233452,-0.0


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-10-19,2026-W43,2026-10-19,20:00,Coventry,Tottenham,17,18,1008.600348,999.300689,0.420521,0.579479,0.233452,-0.0


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-10-19,2026-W43,2026-10-19,20:00,Coventry,Tottenham,17,18,1008.600348,999.300689,0.420521,0.579479,0.233452,-0.0


,rank,team,elo,as_of_date
0,1,Man City,1352.453045,2026-10-19
1,2,Arsenal,1318.928309,2026-10-19
2,3,Brighton,1208.499764,2026-10-19
3,4,Man United,1177.508628,2026-10-19
4,5,Liverpool,1167.033587,2026-10-19
5,6,Brentford,1154.047019,2026-10-19
6,7,Bournemouth,1124.181524,2026-10-19
7,8,Chelsea,1119.950603,2026-10-19
8,9,Leeds,1114.348558,2026-10-19
9,10,Aston Villa,1113.890025,2026-10-19


In [39]:
### Save Data
current_ratings.to_csv("pl_current_elo_rankings.csv", index=False)
daily_projections.to_csv("pl_daily_projections.csv", index=False)
weekly_projections.to_csv("pl_weekly_projections.csv", index=False)
future_projections.to_csv("pl_future_projections.csv", index=False)
# pl_elo["games"].to_csv("pl_elo_history.csv", index=False)